In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             average_precision_score, classification_report)
import matplotlib.pyplot as plt

In [2]:
print("XGBoost version:", xgb.__version__)

XGBoost version: 3.2.0


### 1. Load & explore data

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target        # 569 samples, 30 features; y=1 benign
print(X.shape, "class balance:", np.bincount(y))   # slight imbalance
print(X.describe().T.head())          # quick numeric summary

In [ ]:
X.columns

### Train / validation / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
# Carve a validation set out of train for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

### 3A. scikit-learn API (recommended for most users) early_stopping_rounds & eval_metric are CONSTRUCTOR args

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=2000,          # upper bound; early stopping finds the real number
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",        # AUC-PR: good for mild imbalance
    tree_method="hist",         # default since 2.0; add device="cuda" for GPU
    early_stopping_rounds=50,   # constructor arg in modern XGBoost
    random_state=42,
)
clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
print("Best iteration:", clf.best_iteration)
print("Best score:", clf.best_score)

### Prediction automatically uses the best iteration when early stopping is on


In [ ]:
proba = clf.predict_proba(X_test)[:, 1]
pred  = clf.predict(X_test)
print("Test ACC :", accuracy_score(y_test, pred))
print("Test AUC :", roc_auc_score(y_test, proba))
print("Test AUCPR:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

### 3B. Native (Booster) API with DMatrix — full control

In [ ]:
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval   = xgb.DMatrix(X_val, label=y_val)
dtest  = xgb.DMatrix(X_test, label=y_test)

params = dict(objective="binary:logistic", eval_metric=["logloss", "aucpr"],
              eta=0.05, max_depth=4, min_child_weight=2,
              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
              tree_method="hist", seed=42)

booster = xgb.train(
    params, dtrain, num_boost_round=2000,
    evals=[(dtrain, "train"), (dval, "valid")],
    early_stopping_rounds=50, verbose_eval=100,
)
# With the native API, slice to the best iteration explicitly:
best_range = (0, booster.best_iteration + 1)
native_proba = booster.predict(dtest, iteration_range=best_range)

### 4. Cross-validation with xgb.cv (native API)

In [ ]:
cv = xgb.cv(
    params, dtrain, num_boost_round=2000, nfold=5, stratified=True,
    early_stopping_rounds=50, metrics="aucpr", seed=42, as_pandas=True)
print("CV best rounds:", len(cv), "| CV test AUCPR:",
      cv["test-aucpr-mean"].iloc[-1], "+/-", cv["test-aucpr-std"].iloc[-1])

### 5. Feature importance: gain, weight, cover  (interpret carefully!)

In [ ]:
for imp in ("gain", "weight", "cover"):
    fig, ax = plt.subplots(figsize=(6, 8))
    xgb.plot_importance(clf.get_booster(), importance_type=imp,
                        max_num_features=10, ax=ax, title=f"Importance ({imp})")
    plt.tight_layout(); plt.savefig(f"importance_{imp}.png")

### 6. SHAP values — the preferred, consistent interpretation

In [ ]:
import shap
explainer = shap.TreeExplainer(clf)          # exact TreeSHAP for trees
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, show=False)   # global importance + direction
plt.tight_layout() #; plt.savefig("shap_summary.png")
plt.show()

### 7. Hyperparameter tuning with Optuna (Bayesian / TPE)

In [ ]:
import optuna

def objective(trial):
    param = {
        "objective": "binary:logistic", "eval_metric": "aucpr",
        "tree_method": "hist", "random_state": 42,
        "n_estimators": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "early_stopping_rounds": 50,
    }
    model = xgb.XGBClassifier(**param)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    p = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, p)   # maximize AUC-PR

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)
print("Best params:", study.best_params, "| Best AUCPR:", study.best_value)

### 8. Save & load  (use JSON/UBJSON — stable, portable format)

In [ ]:
clf.save_model("bc_model.json")              # sklearn wrapper
booster.save_model("bc_booster.ubj")         # native, UBJSON (compact/stable)

loaded = xgb.XGBClassifier()
loaded.load_model("bc_model.json")
assert np.allclose(loaded.predict_proba(X_test)[:, 1], proba)
print("Model round-trip OK.")

## Calibration

In [ ]:
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression

proba = clf.predict_proba(X_test)[:, 1]
y_t = np.asarray(y_test)

# (a) calibration-in-the-large: mean prediction vs prevalence
print(f"mean p̂ = {proba.mean():.3f}   prevalence = {y_t.mean():.3f}")

# (b) reliability curve, quantile bins
prob_true, prob_pred = calibration_curve(y_t, proba, n_bins=10, strategy="quantile")
plt.plot(prob_pred, prob_true, "o-"); plt.plot([0,1],[0,1],"k--")  # diagonal = perfect

# (c) ECE with the same binning
def ece(y, p, n_bins=10):
    q = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    b = np.clip(np.searchsorted(q[1:-1], p), 0, n_bins - 1)
    return sum(np.mean(b==k) * abs(y[b==k].mean() - p[b==k].mean())
               for k in range(n_bins) if np.any(b==k))
print("ECE:", ece(y_t, proba))

# (d) calibration slope/intercept on the logit scale
z = np.log(np.clip(proba,1e-6,1-1e-6) / (1 - np.clip(proba,1e-6,1-1e-6)))
lr = LogisticRegression(C=1e6).fit(z.reshape(-1,1), y_t)   # C large ≈ unpenalized
print("slope:", lr.coef_[0,0], "intercept:", lr.intercept_[0])

# (e) proper scores
print("Brier:", brier_score_loss(y_t, proba), " logloss:", log_loss(y_t, proba))

In [ ]:
from sklearn.frozen import FrozenEstimator            # sklearn >= 1.6
from sklearn.calibration import CalibratedClassifierCV

# carve a dedicated calibration split out of the training pool
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=0)

model = xgb.XGBClassifier(**clf.get_params())          # same hyperparameters
model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

cal = CalibratedClassifierCV(FrozenEstimator(model),   # freeze: don't refit the booster
                             method="sigmoid")         # Platt; 114-row calib set is
cal.fit(X_cal, y_cal)                                  # too small for isotonic
# (older sklearn: CalibratedClassifierCV(model, cv="prefit", method="sigmoid"))

p_cal = cal.predict_proba(X_test)[:, 1]
print("Brier before/after:", brier_score_loss(y_t, proba),
      brier_score_loss(y_t, p_cal))
print("AUCPR before/after:", average_precision_score(y_t, proba),
      average_precision_score(y_t, p_cal))   # ~unchanged: the map is monotone

### Check again

In [ ]:
# (a) calibration-in-the-large: mean prediction vs prevalence
print(f"mean p̂ = {p_cal.mean():.3f}   prevalence = {y_t.mean():.3f}")

# (b) reliability curve, quantile bins
prob_true, prob_pred = calibration_curve(y_t, p_cal, n_bins=10, strategy="quantile")
plt.plot(prob_pred, prob_true, "o-"); plt.plot([0,1],[0,1],"k--")  # diagonal = perfect